# 🚗 Vehicle Detection Model Training
## Auto Hotel Luxor - Mexican Market

This notebook trains a MobileNetV3 model for vehicle brand/model/color classification.

**Dataset:** CompCar filtered to 32 Mexican market brands (25K+ images)

**Steps:**
1. Install dependencies
2. Upload dataset
3. Train model
4. Export to TFLite
5. Download model

## 1. Setup - Enable GPU

Go to **Runtime > Change runtime type > T4 GPU**

In [ ]:
# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('⚠️ No GPU detected! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Install dependencies
!pip install -q ultralytics tqdm rich

## 2. Upload Dataset

Upload your `mexican_market.zip` file (the train/val/test folders).

To create the zip locally:
```bash
cd ml-models/datasets
zip -r mexican_market.zip mexican_market/
```

In [ ]:
# Upload dataset
from google.colab import files
import zipfile
import os

print('Upload mexican_market.zip...')
uploaded = files.upload()

# Extract
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f'Extracting {filename}...')
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print('Done!')

# Verify
if os.path.exists('mexican_market/train'):
    brands = os.listdir('mexican_market/train')
    total = sum(len(os.listdir(f'mexican_market/train/{b}')) for b in brands)
    print(f'\n✓ Dataset ready: {len(brands)} brands, {total} training images')
else:
    print('⚠️ Dataset not found. Make sure the zip contains mexican_market/ folder')

## 3. Train Model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
import time

# Dataset class
class VehicleDataset(Dataset):
    def __init__(self, root, transform=None, max_per_class=1000):
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        
        root = Path(root)
        for idx, class_dir in enumerate(sorted([d for d in root.iterdir() if d.is_dir()])):
            self.class_to_idx[class_dir.name] = idx
            imgs = list(class_dir.glob('*.jpg'))[:max_per_class]
            for img in imgs:
                self.images.append(img)
                self.labels.append(idx)
        
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.num_classes = len(self.class_to_idx)
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets
print('Loading datasets...')
train_ds = VehicleDataset('mexican_market/train', train_transform)
val_ds = VehicleDataset('mexican_market/val', val_transform)

print(f'Train: {len(train_ds)} images, {train_ds.num_classes} classes')
print(f'Val: {len(val_ds)} images')
print(f'\nBrands: {list(train_ds.class_to_idx.keys())}')

In [ ]:
# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = models.mobilenet_v3_small(pretrained=True)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, train_ds.num_classes)
model = model.to(device)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

print(f'\nModel: MobileNetV3-Small')
print(f'Classes: {train_ds.num_classes}')
print(f'Batch size: 64')
print(f'Learning rate: 0.001')

In [ ]:
# Training loop
EPOCHS = 15
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print(f'\nTraining for {EPOCHS} epochs...\n')
start_time = time.time()

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*train_correct/train_total:.1f}%'})
    
    train_acc = 100. * train_correct / train_total
    
    # Validate
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100. * val_correct / val_total
    scheduler.step(val_acc)
    
    # Save history
    history['train_loss'].append(train_loss / len(train_loader))
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss / len(val_loader))
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch+1}: Train Loss={train_loss/len(train_loader):.4f} Acc={train_acc:.1f}% | Val Loss={val_loss/len(val_loader):.4f} Acc={val_acc:.1f}%')
    
    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'num_classes': train_ds.num_classes,
            'class_to_idx': train_ds.class_to_idx,
            'idx_to_class': train_ds.idx_to_class,
        }, 'best_model.pth')
        print(f'  ✓ Saved best model (acc: {val_acc:.1f}%)')

elapsed = time.time() - start_time
print(f'\n{"="*50}')
print(f'Training complete!')
print(f'Best accuracy: {best_acc:.1f}%')
print(f'Time: {elapsed/60:.1f} minutes')
print(f'{"="*50}')

## 4. Export to TFLite

In [ ]:
# Export to ONNX then TFLite
!pip install -q onnx onnxruntime

import torch.onnx

# Load best model
checkpoint = torch.load('best_model.pth', map_location='cpu')
model = models.mobilenet_v3_small(pretrained=False)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, checkpoint['num_classes'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Export to ONNX
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model, dummy, 'vehicle_classifier.onnx',
    export_params=True, opset_version=11,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}})
print('✓ Exported to ONNX')

# Save labels
labels = checkpoint['idx_to_class']
with open('vehicle_labels.json', 'w') as f:
    json.dump({str(k): v for k, v in labels.items()}, f, indent=2)
print('✓ Saved labels')

# Convert to TFLite
try:
    import onnx
    from onnx_tf.backend import prepare
    import tensorflow as tf
    
    onnx_model = onnx.load('vehicle_classifier.onnx')
    tf_rep = prepare(onnx_model)
    tf_rep.export_graph('saved_model')
    
    converter = tf.lite.TFLiteConverter.from_saved_model('saved_model')
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    
    with open('vehicle_classifier.tflite', 'wb') as f:
        f.write(tflite_model)
    print(f'✓ Exported to TFLite ({len(tflite_model)/1024/1024:.1f} MB)')
except Exception as e:
    print(f'TFLite export failed: {e}')
    print('Download the ONNX file and convert locally')

## 5. Download Model

In [ ]:
# Download files
from google.colab import files

print('Downloading model files...\n')

# Always download these
files.download('best_model.pth')
files.download('vehicle_labels.json')
files.download('vehicle_classifier.onnx')

# Download TFLite if available
import os
if os.path.exists('vehicle_classifier.tflite'):
    files.download('vehicle_classifier.tflite')

print('\n✓ All files downloaded!')
print('\nNext steps:')
print('1. Copy files to: ml-models/exported/')
print('2. Run: python scripts/05_export_models.py (if TFLite not exported)')
print('3. Bundle with mobile app')

## 6. Test Inference (Optional)

In [ ]:
# Test inference
from PIL import Image
import torchvision.transforms as T

# Load model
checkpoint = torch.load('best_model.pth', map_location='cpu')
model = models.mobilenet_v3_small(pretrained=False)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, checkpoint['num_classes'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

idx_to_class = checkpoint['idx_to_class']

# Test with validation images
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Testing inference on 10 random images...\n')

import random
test_images = random.sample(val_ds.images, 10)

correct = 0
for img_path in test_images:
    img = Image.open(img_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0)
    
    with torch.no_grad():
        output = model(input_tensor)
        _, predicted = output.max(1)
    
    predicted_class = idx_to_class[predicted.item()]
    actual_class = img_path.parent.name
    match = '✓' if predicted_class == actual_class else '✗'
    if predicted_class == actual_class:
        correct += 1
    
    print(f'{match} {actual_class:15s} → {predicted_class:15s} ({img_path.name})')

print(f'\nAccuracy: {correct}/10 ({10*correct}%)')